# Session 3.2 -- Augmented Retrieval & the Embedding Gap

**AI-2: AI Backend Engineering**

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Implement** metadata-filtered retrieval using ChromaDB's `where` clause to restrict results by document type, source, or other metadata fields
2. **Compare** naive vs filtered RAG using a provided comparison utility, measuring retrieval relevance and answer quality
3. **Observe** (via PCA visualization) that questions and answers cluster in different embedding neighborhoods -- the "embedding gap"

## What You Are Working With

- `src/s4_retrieval/filtered.py` -- Filtered retrieval with `filtered_retrieve()`, `filtered_rag()`, and `compare_retrieval()` (provided complete)
- `src/s4_retrieval/naive.py` -- Naive RAG retrieval from Session 3.1 (provided complete)
- `src/s0_generation/generate.py` -- Claude API wrapper (provided complete)
- `src/s2_embeddings/embed.py` -- Voyage AI embedding functions (provided complete)
- `src/s3_ingestion/store.py` -- ChromaDB collection management (provided complete)

You **import and use** the pre-built modules. The `.py` modules are provided as working infrastructure.

## Where We Are in the Pipeline

| Session | What We Built | Status |
|---------|--------------|--------|
| **1.1** | API connection + generation | Done |
| **1.2** | Structured extraction with Pydantic | Done |
| **2.1** | Embeddings + similarity measurement | Done |
| **2.2** | Chunking + vector store ingestion | Done |
| **3.1** | Naive RAG -- retrieve + generate | Done |
| **3.2** | Metadata-filtered RAG + comparison | **Today** |
| **4.1** | Observability and debugging | Next session |

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv()

In [ ]:
import time
import textwrap
import numpy as np

# Naive RAG from Session 3.1
from src.s4_retrieval.naive import naive_retrieve, naive_rag, RAGResponse

# Filtered RAG -- provided complete for this session
from src.s4_retrieval.filtered import filtered_retrieve, filtered_rag, compare_retrieval

# Generation module
from src.s0_generation.generate import call_claude_with_usage

# Embedding module
from src.s2_embeddings.embed import embed_texts, cosine_similarity

# ChromaDB collection
from src.s3_ingestion.store import get_collection

print("All imports loaded successfully.")

In [ ]:
# Verify the ChromaDB collection is populated from Session 2.2
collection = get_collection()
print(f"Collection: {collection.name}")
print(f"Total chunks stored: {collection.count()}")

if collection.count() == 0:
    print("\nWARNING: Collection is empty. Run session_2_2.ipynb first to ingest the corpus.")
else:
    print(f"\nCollection is populated and ready for retrieval.")

---

## 2. Recap: Where Naive RAG Broke

In Session 3.1, you built naive RAG and then you broke it. You found semantic drift, hallucinations, and wrong document types in your results. Let's re-run a few of those failure cases to remind ourselves what went wrong.

The key insight from 3.1: **semantic similarity finds things that are ABOUT the same topic, but "about the same topic" and "useful for answering this specific question" are not the same thing.**

In [ ]:
# Failure Case 1: Wrong document type in results
# Asking about vacation policy -- naive retrieval searches EVERYTHING

query_1 = "What is Northbrook Partners' vacation policy?"
naive_results_1 = naive_retrieve(query_1, top_k=5)

print(f"Query: {query_1}")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in naive_results_1:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

# Count how many different doc_types appear
doc_types = set(r["metadata"].get("doc_type", "?") for r in naive_results_1)
print(f"\nDocument types in results: {doc_types}")
print("Notice: naive retrieval pulls from ANY document type, not just policies.")

In [ ]:
# Failure Case 2: Semantic drift -- topic-adjacent but not useful
# A question about a specific meeting decision pulls in random related content

query_2 = "What decisions were made about remote work at the board meeting?"
naive_results_2 = naive_retrieve(query_2, top_k=5)

print(f"Query: {query_2}")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in naive_results_2:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

print("\nDoes every result actually contain a DECISION about remote work?")
print("Or did some just mention 'remote work' in passing?")

In [ ]:
# Failure Case 3: Financial question pulling non-financial sources

query_3 = "What were the Q3 revenue numbers?"
naive_results_3 = naive_retrieve(query_3, top_k=5)

print(f"Query: {query_3}")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in naive_results_3:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

doc_types = set(r["metadata"].get("doc_type", "?") for r in naive_results_3)
print(f"\nDocument types in results: {doc_types}")
print("Financial data should come from financial documents, not meeting notes.")

**The pattern across all three failures:** Naive retrieval returns whatever is semantically closest, regardless of document type. A meeting note that *mentions* vacation policy scores high, even though the actual policy text has the answer. A memo that references revenue is not the same as the financial report.

**Today's question:** What if we could tell the retrieval system -- *"only look at policy documents"* or *"only look at financial reports"*? That is metadata filtering.

---

## 3. Introducing Metadata Filters

ChromaDB supports metadata filtering at query time via the `where` parameter. This is one line of code -- but it fundamentally changes retrieval behavior.

**Key concept:** Filters run BEFORE similarity search. ChromaDB first narrows the candidate set by metadata, then ranks by similarity within that subset. This is not post-filtering -- you are not retrieving 100 chunks and throwing away 80.

### What metadata is stored on our chunks?

Let's inspect the actual metadata fields available in our ChromaDB collection.

In [ ]:
# Peek at the metadata fields in our collection
collection = get_collection()
sample = collection.peek(limit=10)

print(f"Sample of {len(sample['ids'])} chunks from the collection:\n")

# Show all metadata keys and values
all_keys = set()
for meta in sample["metadatas"]:
    all_keys.update(meta.keys())

print(f"Metadata fields available: {sorted(all_keys)}\n")
print("Sample metadata for each chunk:")
print("-" * 60)
for i, meta in enumerate(sample["metadatas"]):
    print(f"  Chunk {i}: {meta}")

In [ ]:
# What distinct values exist for each metadata field?
# This tells us what filters we can actually build.

# Get a larger sample to see all distinct values
all_data = collection.get(include=["metadatas"], limit=collection.count())

print("Distinct values for each metadata field:\n")
for key in sorted(all_keys):
    if key == "chunk_index":
        # Skip chunk_index -- too many values
        values = set(m.get(key) for m in all_data["metadatas"])
        print(f"  {key:15s}: {len(values)} unique values (0 through {max(values)})")
    else:
        values = sorted(set(str(m.get(key, "")) for m in all_data["metadatas"]))
        print(f"  {key:15s}: {values}")

print("\nYour filters can ONLY use fields that exist in your metadata.")
print("If you did not add 'department' during ingestion, you cannot filter on it now.")

### ChromaDB `where` Clause Syntax

```python
# Exact match
where={"doc_type": "policy"}

# Comparison operators
where={"year": {"$gte": 2024}}

# Logical AND -- both conditions must match
where={
    "$and": [
        {"doc_type": "policy"},
        {"department": "HR"}
    ]
}

# Logical OR -- either condition matches
where={
    "$or": [
        {"doc_type": "policy"},
        {"doc_type": "handbook"}
    ]
}
```

**Important:** Every `$and` condition makes the filter stricter. Two conditions might be fine. Four might give you zero results. Always test.

---

## 4. Exploring `filtered_retrieve()`

The pre-built module `src/s4_retrieval/filtered.py` provides `filtered_retrieve()`. It is almost identical to `naive_retrieve()` -- one extra parameter: `where=filters`. That is it. The entire difference between naive and filtered retrieval is one keyword argument.

Let's explore how different filters change retrieval behavior.

In [ ]:
# Basic filter: only policy documents
query = "What is Northbrook Partners' vacation policy?"

filtered_results = filtered_retrieve(
    query,
    filters={"doc_type": "policy"},
    top_k=5
)

print(f"Query: {query}")
print(f"Filter: doc_type = 'policy'")
print(f"\nResults: {len(filtered_results)} chunks")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in filtered_results:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

print("\nEvery result is a policy document. No meeting notes, no financial reports.")

In [ ]:
# Filter by specific source document
# Useful when you know exactly which document should contain the answer

filtered_by_source = filtered_retrieve(
    query,
    filters={"source": "vacation_policy_2025.md"},
    top_k=5
)

print(f"Query: {query}")
print(f"Filter: source = 'vacation_policy_2025.md'")
print(f"\nResults: {len(filtered_by_source)} chunks")
print(f"\n{'Score':>8}  {'Source':<45}  {'Chunk':>6}")
print("-" * 65)
for r in filtered_by_source:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('chunk_index', '?'):>6}")

print("\nAll results from a single document. Maximum precision.")

In [ ]:
# Filter by document type: meeting notes only
meeting_query = "What decisions were made about remote work at the board meeting?"

filtered_meetings = filtered_retrieve(
    meeting_query,
    filters={"doc_type": "meeting"},
    top_k=5
)

print(f"Query: {meeting_query}")
print(f"Filter: doc_type = 'meeting'")
print(f"\nResults: {len(filtered_meetings)} chunks")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in filtered_meetings:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

In [ ]:
# Using $or to combine document types
# Useful when the answer might be in either policies OR handbooks

filtered_or = filtered_retrieve(
    query,
    filters={
        "$or": [
            {"doc_type": "policy"},
            {"doc_type": "memo"}
        ]
    },
    top_k=5
)

print(f"Query: {query}")
print(f"Filter: doc_type IN ('policy', 'memo')")
print(f"\nResults: {len(filtered_or)} chunks")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in filtered_or:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

### The Over-Filtering Problem

More filters is not always better. Each filter removes candidates. At some point, you remove the right answer.

In [ ]:
# Over-filtering: too restrictive
# CEO statements are in meeting notes, NOT in HR policies

ceo_query = "What did the CEO say about company direction?"

over_filtered = filtered_retrieve(
    ceo_query,
    filters={
        "$and": [
            {"doc_type": "policy"},
            {"source": "vacation_policy_2025.md"}
        ]
    },
    top_k=5
)

print(f"Query: {ceo_query}")
print(f"Filter: doc_type='policy' AND source='vacation_policy_2025.md'")
print(f"\nResults: {len(over_filtered)} chunks")

if len(over_filtered) == 0:
    print("\nZero results. We filtered out the answer.")
    print("The CEO's statements are in meeting notes, not HR policies.")
    print("Over-filtering is WORSE than no filtering at all.")
else:
    for r in over_filtered:
        print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')}")

In [ ]:
# Compare: the same CEO query with a correct filter
correctly_filtered = filtered_retrieve(
    ceo_query,
    filters={"doc_type": "meeting"},
    top_k=5
)

print(f"Query: {ceo_query}")
print(f"Filter: doc_type = 'meeting'")
print(f"\nResults: {len(correctly_filtered)} chunks")
print(f"\n{'Score':>8}  {'Source':<45}")
print("-" * 55)
for r in correctly_filtered:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}")

print("\nCorrect filter type = relevant results.")
print("The engineering question: how do you CHOOSE the right filter for a query?")

**Takeaway:** Filtering is a precision tool. Applied correctly, it removes noise. Applied incorrectly, it removes the answer.

- **Filtering helps when** the question implies a specific document type, time period, or category
- **Filtering hurts when** the filter is too restrictive, the answer spans multiple document types, or the metadata is inconsistent

**The precision/recall tradeoff:**
- *Naive RAG:* high recall (considers everything, unlikely to miss relevant chunks), lower precision (includes irrelevant chunks too)
- *Filtered RAG:* higher precision (everything matches the filter), risk of lower recall (might filter out relevant chunks)

---

## 5. Fixing the Failures

Let's go back to the three failure cases from Section 2 and fix them with appropriate metadata filters. For each, we will run both naive and filtered retrieval side by side and compare the sources.

In [ ]:
# Helper function to display side-by-side retrieval results
def show_comparison(query, filters, filter_description):
    """Run naive and filtered retrieval and display results side by side."""
    naive_results = naive_retrieve(query, top_k=5)
    filtered_results = filtered_retrieve(query, filters=filters, top_k=5)

    print(f"Query: {query}")
    print(f"Filter: {filter_description}")
    print()

    print("=== NAIVE RETRIEVAL ===")
    for r in naive_results:
        print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?'):<40} type={r['metadata'].get('doc_type', '?')}")

    print()
    print("=== FILTERED RETRIEVAL ===")
    if not filtered_results:
        print("  [No results -- filter too restrictive]")
    else:
        for r in filtered_results:
            print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?'):<40} type={r['metadata'].get('doc_type', '?')}")

    # Show overlap
    naive_texts = {r["text"][:100] for r in naive_results}
    filtered_texts = {r["text"][:100] for r in filtered_results}
    overlap = len(naive_texts & filtered_texts)
    print(f"\nOverlap: {overlap}/{min(len(naive_results), len(filtered_results))} chunks in common")
    print("-" * 75)

In [ ]:
# Fix Failure Case 1: Wrong document type
# Original problem: naive retrieval pulled meeting notes and memos
# for a question about vacation POLICY

show_comparison(
    query="What is Northbrook Partners' vacation policy?",
    filters={"doc_type": "policy"},
    filter_description="doc_type = 'policy'"
)

print("\nFiltered: all results from policy documents. No noise from meeting notes.")

In [ ]:
# Fix Failure Case 2: Semantic drift
# Original problem: any document mentioning 'remote work' appeared
# Fix: restrict to meeting documents where actual decisions live

show_comparison(
    query="What decisions were made about remote work at the board meeting?",
    filters={"doc_type": "meeting"},
    filter_description="doc_type = 'meeting'"
)

print("\nFiltered: only meeting notes. The actual decisions, not policy text that mentions remote work.")

In [ ]:
# Fix Failure Case 3: Financial data from non-financial sources
# Original problem: meeting notes mentioning revenue appeared alongside
# actual financial documents

show_comparison(
    query="What were the Q3 revenue numbers?",
    filters={"doc_type": "financial"},
    filter_description="doc_type = 'financial'"
)

print("\nFiltered: only financial documents. Actual numbers, not secondhand references.")

**Pattern across all three fixes:**

1. The naive top score might be higher, but the filtered results are from the *right kind of document*
2. A 0.72 from the right document type often produces a better answer than a 0.85 from an irrelevant document
3. The score measures similarity, not usefulness

---

## 6. Naive vs Filtered: Full RAG Comparison

Retrieval is only half the story. Let's compare the actual *answers* generated by both pipelines. The `compare_retrieval()` function from `filtered.py` runs both pipelines on the same query and shows us:
- Retrieval overlap (how many chunks are the same)
- Score differences
- Source differences

We will also generate answers from both to see how retrieval quality affects the final output.

In [ ]:
# Use compare_retrieval() to compare both pipelines
comparison = compare_retrieval(
    "What is Northbrook Partners' vacation policy?",
    filters={"doc_type": "policy"},
    top_k=5
)

print(f"Query: {comparison['question']}")
print(f"Filter: {comparison['filters']}")
print(f"\nNaive top score:    {comparison['naive_top_score']:.3f}")
print(f"Filtered top score: {comparison['filtered_top_score']:.3f}")
print(f"Chunk overlap:      {comparison['overlap']} of 5 chunks in common")

# Show naive sources
print("\nNaive sources:")
for s in comparison["naive_sources"]:
    print(f"  [{s['score']:.3f}] {s['metadata'].get('source', '?')} ({s['metadata'].get('doc_type', '?')})")

# Show filtered sources
print("\nFiltered sources:")
for s in comparison["filtered_sources"]:
    print(f"  [{s['score']:.3f}] {s['metadata'].get('source', '?')} ({s['metadata'].get('doc_type', '?')})")

In [ ]:
# Now let's compare the actual ANSWERS from both pipelines
# This is where the quality difference becomes visible

query = "What is Northbrook Partners' vacation policy?"

print("Generating answers from both pipelines...\n")

# Naive RAG -- full pipeline
naive_response = naive_rag(query, top_k=5)

# Filtered RAG -- full pipeline
filtered_response = filtered_rag(query, filters={"doc_type": "policy"}, top_k=5)

print("=" * 70)
print("NAIVE RAG ANSWER")
print("=" * 70)
print(naive_response.answer)
print(f"\nSources used: {len(naive_response.sources)}")
print(f"Tokens: {naive_response.input_tokens} in / {naive_response.output_tokens} out")

print()
print("=" * 70)
print("FILTERED RAG ANSWER")
print("=" * 70)
print(filtered_response.answer)
print(f"\nSources used: {len(filtered_response.sources)}")
print(f"Tokens: {filtered_response.input_tokens} in / {filtered_response.output_tokens} out")

print()
print("Compare: Which answer is more focused? Which cites better sources?")

In [ ]:
# Run a batch of comparisons across different query types
# This is the pattern you will use for Lab 2

evaluation_queries = [
    {
        "query": "What is Northbrook Partners' vacation policy?",
        "filters": {"doc_type": "policy"},
        "intent": "policy_lookup",
        "expected_source_type": "policy",
        "notes": "Filtering should help -- only policy docs have the official answer"
    },
    {
        "query": "What decisions were made about remote work at the board meeting?",
        "filters": {"doc_type": "meeting"},
        "intent": "meeting_search",
        "expected_source_type": "meeting",
        "notes": "Filtering should help -- decisions are in meeting notes"
    },
    {
        "query": "What were the Q3 revenue numbers?",
        "filters": {"doc_type": "financial"},
        "intent": "financial_query",
        "expected_source_type": "financial",
        "notes": "Filtering should help -- revenue data lives in financial reports"
    },
    {
        "query": "What services does Northbrook Partners offer?",
        "filters": {"doc_type": "memo"},
        "intent": "general",
        "expected_source_type": "multiple",
        "notes": "Filtering might NOT help -- services mentioned across doc types"
    },
    {
        "query": "What is the professional development budget?",
        "filters": {"doc_type": "policy"},
        "intent": "policy_lookup",
        "expected_source_type": "policy",
        "notes": "Tests whether budget info is in policies or financial docs"
    },
]

print(f"Running {len(evaluation_queries)} comparisons...\n")
print(f"{'#':>3}  {'Intent':<18}  {'Naive':>8}  {'Filtered':>10}  {'Overlap':>8}  Notes")
print("-" * 90)

comparison_results = []
for i, eq in enumerate(evaluation_queries, 1):
    result = compare_retrieval(eq["query"], filters=eq["filters"], top_k=5)
    result["intent"] = eq["intent"]
    result["notes"] = eq["notes"]
    comparison_results.append(result)

    naive_types = set(s["metadata"].get("doc_type", "?") for s in result["naive_sources"])
    filtered_types = set(s["metadata"].get("doc_type", "?") for s in result["filtered_sources"])

    print(f"{i:>3}  {eq['intent']:<18}  {result['naive_top_score']:>8.3f}  {result['filtered_top_score']:>10.3f}  {result['overlap']:>8}  {eq['notes'][:40]}")

print("\nComparison complete. Results stored in comparison_results for further analysis.")

### Design Your Own Queries

The evaluation queries above are a starting point. For Lab 2, you need at least 5 diverse queries. Add your own below.

**Good test query checklist:**
- Has a **predictable answer path** (you know what doc type should contain the answer)
- Tests a specific **intent** (policy lookup, financial query, meeting search, etc.)
- Includes at least one query where **filtering should NOT help** (to demonstrate the tradeoff)

In [ ]:
# YOUR QUERIES: Add 2-3 of your own evaluation queries below
# Think about the Northbrook corpus: policies, meeting notes, financial reports, memos

my_queries = [
    {
        "query": "YOUR QUESTION HERE",
        "filters": {"doc_type": "YOUR_FILTER"},
        "intent": "your_intent",
        "notes": "Why this query is interesting"
    },
    # Add more queries...
]

# Run your custom queries through compare_retrieval()
for i, q in enumerate(my_queries, 1):
    if q["query"] == "YOUR QUESTION HERE":
        print("Edit the queries above before running this cell!")
        break
    result = compare_retrieval(q["query"], filters=q["filters"], top_k=5)
    naive_types = [s["metadata"].get("doc_type", "?") for s in result["naive_sources"]]
    filtered_types = [s["metadata"].get("doc_type", "?") for s in result["filtered_sources"]]
    print(f"\nQuery {i}: {q['query']}")
    print(f"  Filter: {q['filters']}")
    print(f"  Naive:    top={result['naive_top_score']:.3f}, types={naive_types}")
    print(f"  Filtered: top={result['filtered_top_score']:.3f}, types={filtered_types}")
    print(f"  Overlap:  {result['overlap']} chunks in common")
    print(f"  Notes:    {q['notes']}")

In [ ]:
# Detailed view: for each comparison, show what sources each pipeline returned
for i, result in enumerate(comparison_results, 1):
    print(f"\n{'=' * 70}")
    print(f"Query {i}: {result['question']}")
    print(f"Intent: {result['intent']} | Filter: {result['filters']}")
    print(f"Overlap: {result['overlap']} chunks in common")

    # Naive sources
    naive_types = [s["metadata"].get("doc_type", "?") for s in result["naive_sources"]]
    print(f"\n  Naive sources ({result['naive_top_score']:.3f}): types = {naive_types}")
    for s in result["naive_sources"]:
        print(f"    [{s['score']:.3f}] {s['metadata'].get('source', '?')}")

    # Filtered sources
    filtered_types = [s["metadata"].get("doc_type", "?") for s in result["filtered_sources"]]
    print(f"\n  Filtered sources ({result['filtered_top_score']:.3f}): types = {filtered_types}")
    if not result["filtered_sources"]:
        print("    [No results -- filter too restrictive]")
    else:
        for s in result["filtered_sources"]:
            print(f"    [{s['score']:.3f}] {s['metadata'].get('source', '?')}")

print(f"\n{'=' * 70}")

**What to notice in the results:**

| Signal | What It Means |
|--------|---------------|
| Filtered sources all same type | Filter is working correctly |
| Naive has mixed source types | Unfiltered retrieval pulls from everywhere |
| Filtered returns 0 chunks | Over-filtering -- too restrictive |
| Same sources in both | Filter is not relevant for this query |
| Lower filtered score, better answer | Score is not the whole story |

---

## 7. PCA Visualization: The Embedding Gap

Before we move on, there is something fundamental about **all** similarity-based retrieval -- naive and filtered alike -- that you need to see.

We are going to embed all the chunks in the corpus, project them into 2D (or 3D) using PCA, and then embed some questions to see where they land.

**The insight:** Questions and answers live in different neighborhoods of embedding space. This is not something filtering fixes.

In [ ]:
# Load the pre-computed PCA transformer from the instructor demo
# This ensures students see the SAME coordinate system as the instructor's visualization

import pickle
from pathlib import Path
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca_path = Path("pca_transformer.pkl")

if pca_path.exists():
    with open(pca_path, "rb") as f:
        pca_data = pickle.load(f)

    pca = pca_data["pca"]
    chunk_pca = pca_data["chunk_coords_3d"]
    doc_types = pca_data["chunk_doc_types"]
    chunk_sources = pca_data["chunk_sources"]
    chunk_texts = pca_data["chunk_texts"]
    chunk_embeddings = pca_data["chunk_embeddings"]
    color_map = pca_data["doc_type_colors"]
    unique_types = pca_data["doc_type_names"]

    # For compatibility with Plotly hover text
    chunk_metadatas = [{"doc_type": dt, "source": src} for dt, src in zip(doc_types, chunk_sources)]

    print(f"Loaded PCA transformer from {pca_path}")
    print(f"  Chunks: {len(chunk_pca)}")
    print(f"  Document types: {unique_types}")
    print(f"  Variance explained: {pca.explained_variance_ratio_.sum():.1%}")
else:
    print(f"PCA pickle not found at {pca_path}")
    print("Falling back to computing from ChromaDB...")

    collection = get_collection()
    all_data = collection.get(
        include=["embeddings", "metadatas", "documents"],
        limit=collection.count()
    )

    chunk_embeddings = np.array(all_data["embeddings"])
    chunk_metadatas = all_data["metadatas"]
    chunk_texts = all_data["documents"]

    doc_types = [m.get("doc_type", "unknown") for m in chunk_metadatas]
    unique_types = sorted(set(doc_types))
    chunk_sources = [m.get("source", "unknown") for m in chunk_metadatas]

    pca = PCA(n_components=3)
    chunk_pca = pca.fit_transform(chunk_embeddings)

    color_map = {
        "policy": "#3b82f6",
        "meeting": "#22c55e",
        "financial": "#f97316",
        "memo": "#a855f7",
        "unknown": "#6b7280",
    }

    print(f"Computed PCA from {len(chunk_embeddings)} chunks")
    print(f"  Document types: {unique_types}")
    print(f"  Variance explained: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
# Display PCA variance information
# The PCA transformer captures a fraction of the total variance --
# think of it like a photograph of a sculpture: you can see the shape,
# but you lose some detail from the angles you cannot see.

print(f"PCA: {chunk_embeddings.shape[1] if hasattr(chunk_embeddings, 'shape') else '?'} dims → 3 dims")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.1%}")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.1%}")
print(f"  PC3: {pca.explained_variance_ratio_[2]:.1%}")
print(f"\nUsing {'pre-computed' if pca_path.exists() else 'freshly computed'} PCA transformer.")
print("New questions will be projected into the same coordinate space as the instructor demo.")

In [ ]:
# Step 3: Create the base 2D scatter plot of corpus chunks by doc_type
# Using the first two PCA components

color_map = {
    "policy": "#3b82f6",     # blue
    "meeting": "#22c55e",    # green
    "financial": "#f97316",  # orange
    "memo": "#a855f7",      # purple
    "unknown": "#6b7280",   # gray
}

fig, ax = plt.subplots(figsize=(12, 8))

for doc_type in unique_types:
    mask = [dt == doc_type for dt in doc_types]
    indices = [i for i, m in enumerate(mask) if m]
    color = color_map.get(doc_type, "#6b7280")
    ax.scatter(
        chunk_pca[indices, 0],
        chunk_pca[indices, 1],
        c=color,
        label=doc_type,
        alpha=0.6,
        s=40,
        edgecolors="white",
        linewidth=0.5,
    )

ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_title("Northbrook Corpus Chunks in Embedding Space (PCA)")
ax.legend(title="Document Type")
plt.tight_layout()
plt.show()

print("See the clusters? Similar content lives in similar regions.")
print("This is why semantic search works at all.")

In [ ]:
# Step 4: Now embed some questions and see where they land
# This is the key insight: questions do NOT land on top of their answer chunks

demo_questions = [
    "What is Northbrook's vacation policy?",
    "What were the Q3 revenue numbers?",
    "What decisions were made at the board meeting?",
    "How does the performance review process work?",
    "What is the remote work policy?",
]

print("Embedding questions...")
question_embeddings = embed_texts(demo_questions)
question_pca = pca.transform(np.array(question_embeddings))

# Plot chunks + questions together
fig, ax = plt.subplots(figsize=(12, 8))

# Plot corpus chunks (smaller, transparent)
for doc_type in unique_types:
    mask = [dt == doc_type for dt in doc_types]
    indices = [i for i, m in enumerate(mask) if m]
    color = color_map.get(doc_type, "#6b7280")
    ax.scatter(
        chunk_pca[indices, 0],
        chunk_pca[indices, 1],
        c=color,
        label=f"chunks: {doc_type}",
        alpha=0.4,
        s=30,
        edgecolors="white",
        linewidth=0.5,
    )

# Plot questions as large red markers
ax.scatter(
    question_pca[:, 0],
    question_pca[:, 1],
    c="#ef4444",
    marker="X",
    s=200,
    label="QUESTIONS",
    edgecolors="black",
    linewidth=1.5,
    zorder=5,
)

# Label each question
for i, q in enumerate(demo_questions):
    # Truncate long labels
    label = q[:35] + "..." if len(q) > 35 else q
    ax.annotate(
        label,
        (question_pca[i, 0], question_pca[i, 1]),
        fontsize=7,
        ha="left",
        va="bottom",
        xytext=(5, 5),
        textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="#fef2f2", alpha=0.8),
    )

ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_title("The Embedding Gap: Questions vs Answer Chunks")
ax.legend(title="Type", loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

print("\nRed X markers = questions. Colored dots = corpus chunks.")
print("\nNotice: questions cluster TOGETHER, away from the answer chunks.")
print("The question 'What is the vacation policy?' is closer to OTHER questions")
print("than to the policy chunks that contain the actual answer.")

In [ ]:
# Step 5: Try 3D visualization with Plotly (if available)
# Falls back gracefully to 2D matplotlib if Plotly is not installed

try:
    import plotly.graph_objects as go

    # Build traces for each document type
    traces = []
    for doc_type in unique_types:
        mask = [dt == doc_type for dt in doc_types]
        indices = [i for i, m in enumerate(mask) if m]
        color = color_map.get(doc_type, "#6b7280")

        hover_texts = [
            f"Source: {chunk_metadatas[j].get('source', '?')}<br>"
            f"Type: {chunk_metadatas[j].get('doc_type', '?')}<br>"
            f"Chunk: {chunk_metadatas[j].get('chunk_index', '?')}<br>"
            f"Text: {chunk_texts[j][:80]}..."
            for j in indices
        ]

        traces.append(go.Scatter3d(
            x=chunk_pca[indices, 0],
            y=chunk_pca[indices, 1],
            z=chunk_pca[indices, 2],
            mode="markers",
            name=f"chunks: {doc_type}",
            marker=dict(size=4, color=color, opacity=0.6),
            hovertext=hover_texts,
            hoverinfo="text",
        ))

    # Add question markers
    question_hover = [f"QUESTION: {q}" for q in demo_questions]
    traces.append(go.Scatter3d(
        x=question_pca[:, 0],
        y=question_pca[:, 1],
        z=question_pca[:, 2],
        mode="markers+text",
        name="QUESTIONS",
        marker=dict(size=8, color="#ef4444", symbol="x", opacity=1.0),
        text=[q[:30] + "..." for q in demo_questions],
        textposition="top center",
        textfont=dict(size=8),
        hovertext=question_hover,
        hoverinfo="text",
    ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title="The Embedding Gap: Questions vs Answer Chunks (3D PCA)",
        scene=dict(
            xaxis_title="PC1",
            yaxis_title="PC2",
            zaxis_title="PC3",
        ),
        width=900,
        height=650,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    )
    fig.show()

    print("\nRotate the 3D plot to see the gap from different angles.")
    print("Hover over points to see which document/question they represent.")

except ImportError:
    print("Plotly not installed -- 2D matplotlib visualization shown above is sufficient.")
    print("Install with: pip install plotly   (optional, for interactive 3D exploration)")

### Understanding the Embedding Gap

What you are seeing:

- **Corpus chunks** cluster by document type. Policy chunks are near other policy chunks. Meeting notes cluster together.
- **Questions** cluster together, in a *different* region from the answer chunks.
- The question "What is the vacation policy?" is semantically closer to *other questions about vacation* than to the statement "Employees accrue 15 days of PTO per year."

**Why?** Questions have interrogative structure. Answers have factual statement structure. Different structures produce different embedding signatures.

**What this means for RAG:**
- Semantic similarity can bridge small gaps -- that is why RAG works at all
- But sometimes the gap is too wide, and the question is closer to a document that *mentions* the topic than to the one that *answers* it
- **Filtering does not fix this.** Filtering narrows the candidate pool, but within that pool, the gap is still there

**In AI-3, you will learn two techniques to close this gap:**
1. **HyDE** -- transform the query to sound like a hypothetical answer before searching
2. **Question Enrichment** -- add "what questions does this chunk answer?" to chunks at ingestion time

For now, just see the problem. Hold onto this insight.

In [ ]:
# YOUR TURN: Add your own question to the embedding space
# Type any question you might ask the Northbrook corpus.
# Where does it land? Near the answer chunks, or off in question space?

# ----- CHANGE THIS -----
my_question = "What is the dress code at Northbrook Partners?"
# ----- CHANGE THIS -----

# Embed and project the question
my_q_embedding = embed_texts([my_question])[0]
my_q_pca = pca.transform(np.array([my_q_embedding]))

# Plot with the original questions + your new question
fig, ax = plt.subplots(figsize=(12, 8))

# Corpus chunks
for doc_type in unique_types:
    mask = [dt == doc_type for dt in doc_types]
    indices = [i for i, m in enumerate(mask) if m]
    color = color_map.get(doc_type, "#6b7280")
    ax.scatter(
        chunk_pca[indices, 0],
        chunk_pca[indices, 1],
        c=color,
        label=f"chunks: {doc_type}",
        alpha=0.4,
        s=30,
        edgecolors="white",
        linewidth=0.5,
    )

# Original demo questions (smaller red)
ax.scatter(
    question_pca[:, 0],
    question_pca[:, 1],
    c="#ef4444",
    marker="X",
    s=120,
    label="Demo questions",
    edgecolors="black",
    linewidth=1,
    alpha=0.6,
    zorder=5,
)

# YOUR question (large, distinct color)
ax.scatter(
    my_q_pca[0, 0],
    my_q_pca[0, 1],
    c="#fbbf24",
    marker="*",
    s=400,
    label="YOUR question",
    edgecolors="black",
    linewidth=1.5,
    zorder=10,
)
ax.annotate(
    my_question[:40] + ("..." if len(my_question) > 40 else ""),
    (my_q_pca[0, 0], my_q_pca[0, 1]),
    fontsize=8,
    fontweight="bold",
    ha="left",
    va="bottom",
    xytext=(8, 8),
    textcoords="offset points",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#fef9c3", alpha=0.9),
)

ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_title("Where Does YOUR Question Land?")
ax.legend(title="Type", loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

print(f"\nYour question: '{my_question}'")
print(f"PCA coordinates: ({my_q_pca[0, 0]:.3f}, {my_q_pca[0, 1]:.3f}, {my_q_pca[0, 2]:.3f})")
print("\nDid it land near the answer chunks, or in the question neighborhood?")
print("Try changing the question and re-running this cell.")

In [ ]:
# Measure the gap numerically: average distance from each question
# to the nearest chunk of the "expected" type vs nearest chunk of any type

print("Nearest chunk distance for each demo question:\n")
print(f"{'Question':<45}  {'Nearest ANY':>12}  {'Nearest Expected':>17}  {'Gap':>6}")
print("-" * 85)

expected_types = ["policy", "financial", "meeting", "policy", "policy"]

for q, q_emb, expected in zip(demo_questions, question_embeddings, expected_types):
    # Cosine similarity to all chunks
    sims = [cosine_similarity(q_emb, chunk_emb.tolist()) for chunk_emb in chunk_embeddings]

    # Nearest chunk of any type
    best_any = max(sims)

    # Nearest chunk of the expected type
    expected_sims = [s for s, dt in zip(sims, doc_types) if dt == expected]
    best_expected = max(expected_sims) if expected_sims else 0.0

    gap = best_any - best_expected
    label = q[:43] + ".." if len(q) > 43 else q
    print(f"{label:<45}  {best_any:>12.4f}  {best_expected:>17.4f}  {gap:>+6.4f}")

print("\nPositive gap = the nearest chunk overall is NOT of the expected type.")
print("This is the embedding gap in numbers.")

---

## 8. The Blind Test

Here is a quick exercise: you see two answers to the same question -- Pipeline A and Pipeline B. You do not know which is naive and which is filtered. Read both. Decide which is better. Then reveal.

This removes confirmation bias. If you know which pipeline produced the answer, you might unconsciously favor the one you expect to be better.

> **Instructor note:** This blind test is conditional. It is most effective when prototyping has confirmed 2-3 queries where filtered retrieval produces noticeably better answers than naive. If the quality lift is not clear for your corpus, skip this section and proceed to Section 9 (Bridge to Observability).

In [ ]:
import random

def blind_test(query, filters, seed=None):
    """Run a blind comparison: show two answers without revealing which pipeline produced which."""
    if seed is not None:
        random.seed(seed)

    # Generate both answers
    naive_response = naive_rag(query, top_k=5)
    filtered_response = filtered_rag(query, filters=filters, top_k=5)

    # Randomly assign to A and B
    if random.random() > 0.5:
        answer_a, answer_b = naive_response, filtered_response
        reveal = {"A": "NAIVE", "B": "FILTERED"}
    else:
        answer_a, answer_b = filtered_response, naive_response
        reveal = {"A": "FILTERED", "B": "NAIVE"}

    print(f"Question: {query}")
    print()
    print("=" * 70)
    print("ANSWER A")
    print("=" * 70)
    print(answer_a.answer)
    print(f"\n  Sources: {len(answer_a.sources)} | Tokens: {answer_a.input_tokens}+{answer_a.output_tokens}")

    print()
    print("=" * 70)
    print("ANSWER B")
    print("=" * 70)
    print(answer_b.answer)
    print(f"\n  Sources: {len(answer_b.sources)} | Tokens: {answer_b.input_tokens}+{answer_b.output_tokens}")

    print()
    print("Which answer is better? A or B?")
    print("(Think about: focus, accuracy, source citation, completeness)")

    return reveal

In [ ]:
# Run the blind test
reveal = blind_test(
    "What is Northbrook Partners' vacation policy?",
    filters={"doc_type": "policy"},
    seed=42
)

In [ ]:
# REVEAL: Which pipeline produced which answer?
# Only run this cell AFTER you have decided which is better.

print("REVEAL:")
print(f"  Answer A was: {reveal['A']}")
print(f"  Answer B was: {reveal['B']}")
print()
print("Were you right? Did the better answer come from the pipeline you expected?")

In [ ]:
# Try another blind test with a different query
# Change the query and filter to test different scenarios

reveal_2 = blind_test(
    "What decisions were made about remote work at the board meeting?",
    filters={"doc_type": "meeting"},
    seed=99
)

In [ ]:
# REVEAL
print("REVEAL:")
print(f"  Answer A was: {reveal_2['A']}")
print(f"  Answer B was: {reveal_2['B']}")

---

## 9. Bridge to Observability

We can now build and compare two retrieval pipelines. But consider:

- You ran 5 queries today and read every answer. What if you had 100 queries? 1,000?
- How would you know if your pipeline started giving subtly wrong answers?
- If the answer is bad, how do you figure out what went wrong -- was it retrieval? Generation? The prompt? The data?

You need **visibility into your pipeline.** Logging, metrics, and tracing that tell you what happened at every step.

**Next session (4.1):** Observability and Debugging. You will build a `PipelineLogger` that captures:
- Query and retrieval results (sources, scores)
- Latency at each pipeline stage
- Token usage and cost tracking
- Answer quality signals

The `PipelineLogger` from Session 4.1 is a **required component of Lab 2.**

### Questions to Think About

> **On Visibility:** You have built the pipeline -- how do you know it is working? What would you log?

> **On Debugging:** When your RAG pipeline gives a bad answer, how do you figure out what went wrong? Was it retrieval? Generation? The prompt? The data?

> **On Production:** What metrics would tell you your pipeline is healthy? Degrading? What is the difference between a pipeline that is down and a pipeline that is silently giving bad answers?

---

## 10. Session Recap

### What we explored today

1. **Revisited naive RAG failures** -- wrong document types, semantic drift, irrelevant sources
2. **ChromaDB `where` clause** -- one keyword argument that fundamentally changes retrieval behavior
3. **Filtered retrieval exploration** -- `filtered_retrieve()` with different filter combinations (doc_type, source, $or, $and)
4. **The over-filtering problem** -- more filters is not always better; empty results are worse than noisy results
5. **Side-by-side comparison** -- naive vs filtered on the same queries, seeing how sources and answers differ
6. **PCA visualization** -- the embedding gap between questions and answer chunks; a fundamental property of embedding space
7. **Blind test** -- comparing answers without knowing which pipeline produced them

### Key takeaways

- **Metadata filtering is one parameter** (`where=`) that changes retrieval from "search everything" to "search within a category"
- **Filters run BEFORE similarity search** -- ChromaDB narrows first, then ranks by similarity
- **Lower similarity scores in filtered results can still produce better answers** -- score measures similarity, not usefulness
- **Over-filtering returns zero results** -- worse than naive retrieval, which at least returns something
- **The embedding gap is real** -- questions and answers cluster in different neighborhoods; filtering does not fix this; AI-3 addresses it with HyDE and question enrichment

### Lab 2 -- Assigned Today

**Lab 2: RAG Pipeline Evaluation -- Naive vs Metadata-Aware**

| Field | Detail |
|-------|--------|
| **Due** | Start of Session 4.2 (two weeks) |
| **Weight** | 20% of course grade |
| **Foundation** | Today's in-class code |

**What you submit:**
1. Both pipeline implementations -- naive and metadata-filtered RAG, working correctly
2. Test query set -- 5+ queries run against both pipelines, results documented
3. Comparison report -- for each query: retrieved chunks, similarity scores, generated answers
4. Analysis -- identify at least 2 cases where metadata filtering improved results; explain WHY
5. Metrics -- integrate PipelineLogger from Session 4.1 for latency, scores, and quality metrics

**Rubric:**

| Criterion | Weight |
|-----------|--------|
| Pipeline Implementation | 25 pts |
| Test Coverage | 25 pts |
| Comparison Analysis | 25 pts |
| Metrics & Observability | 25 pts |

**Start from today's code.** Refine your queries and filters. Run the full comparison. Then focus on the report -- the analysis is where the points are.

The Metrics criterion requires PipelineLogger from Session 4.1 -- we build that next session. Plan for it.

### Next Session: 4.1 -- Observability and Debugging

You have built the whole pipeline. Now the question is: how do you see inside it?